# Commodities Market Response: Ship Movement + AI Sentiment Analysis

Analyzing how commodities (Oil, Gas, Shipping) respond to:
- Ship traffic disruptions in Strait of Hormuz
- AI-analyzed geopolitical sentiment
- Combined impact of both factors

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

plt.rcParams['figure.figsize'] = (20, 12)
plt.rcParams['font.size'] = 11

## 1. Load Data

In [2]:
# Load shipping data
shipping_df = pd.read_csv('Data/Portwatch_Shipment_Data/arrivals-of-ships.csv')
shipping_df['DateTime'] = pd.to_datetime(shipping_df['DateTime'])
shipping_df.set_index('DateTime', inplace=True)
shipping_df['Total'] = shipping_df[['Container', 'Dry Bulk', 'General Cargo', 'Roll-on/roll-off', 'Tanker']].sum(axis=1)

# Load sentiment data
sentiment_df = pd.read_csv('Data/crisis_sentiment_analysis.csv')
sentiment_df['date'] = pd.to_datetime(sentiment_df['date'])

sentiment_daily = pd.read_csv('Data/crisis_sentiment_daily.csv')
sentiment_daily['date'] = pd.to_datetime(sentiment_daily['date'])

# Load commodities data
commodities = {}
for ticker in ['USO', 'BNO', 'UNG', 'BDRY']:
    df = pd.read_parquet(f'Data/Commodities/{ticker}.parquet')
    df.index = pd.to_datetime(df.index)
    commodities[ticker] = df['Close']

# Load futures data
for ticker in ['CL', 'BZ', 'NG']:
    df = pd.read_parquet(f'Data/Futures/{ticker}.parquet')
    df.index = pd.to_datetime(df.index)
    commodities[f'{ticker}_Fut'] = df['Close']

commodities_df = pd.DataFrame(commodities)

print(f"Shipping data: {len(shipping_df)} days")
print(f"Sentiment articles: {len(sentiment_df)}")
print(f"Commodities: {list(commodities_df.columns)}")
print(f"Date range: {commodities_df.index.min()} to {commodities_df.index.max()}")

ValueError: If using all scalar values, you must pass an index

## 2. Calculate Shipping Disruption Metric

In [ ]:
# Calculate shipping disruption (deviation from moving average)
shipping_df['MA_30'] = shipping_df['Total'].rolling(30).mean()
shipping_df['Disruption_Pct'] = ((shipping_df['Total'] - shipping_df['MA_30']) / shipping_df['MA_30']) * 100
shipping_df['Tanker_Disruption_Pct'] = ((shipping_df['Tanker'] - shipping_df['Tanker'].rolling(30).mean()) / 
                                         shipping_df['Tanker'].rolling(30).mean()) * 100

# Visualize shipping disruption
fig, axes = plt.subplots(2, 1, figsize=(20, 10))

# Total traffic
axes[0].plot(shipping_df.index, shipping_df['Total'], label='Daily Total', alpha=0.6)
axes[0].plot(shipping_df.index, shipping_df['MA_30'], label='30-Day MA', linewidth=2, color='red')
axes[0].set_title('Ship Arrivals - Total Traffic', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Ships')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Disruption percentage
axes[1].plot(shipping_df.index, shipping_df['Disruption_Pct'], color='orange', alpha=0.7)
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1].axhline(y=-20, color='red', linestyle='--', linewidth=1, label='Crisis Threshold (-20%)')
axes[1].fill_between(shipping_df.index, shipping_df['Disruption_Pct'], 0, 
                      where=shipping_df['Disruption_Pct'] < -20, color='red', alpha=0.3)
axes[1].set_title('Shipping Disruption (% Deviation from 30-Day MA)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Disruption %')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nShipping Disruption Statistics:")
print(f"Mean disruption: {shipping_df['Disruption_Pct'].mean():.2f}%")
print(f"Std deviation: {shipping_df['Disruption_Pct'].std():.2f}%")
print(f"Days with >20% disruption: {(shipping_df['Disruption_Pct'] < -20).sum()}")

## 3. Merge All Data

In [ ]:
# Create daily sentiment score
sentiment_daily_indexed = sentiment_daily.set_index('date')

# Merge all data
analysis_df = commodities_df.copy()
analysis_df = analysis_df.join(shipping_df[['Total', 'Tanker', 'Disruption_Pct', 'Tanker_Disruption_Pct']], how='left')
analysis_df = analysis_df.join(sentiment_daily_indexed[['sentiment_mean', 'article_count', 'risk_level']], how='left')

# Forward fill shipping data (only available for certain dates)
analysis_df['Disruption_Pct'] = analysis_df['Disruption_Pct'].fillna(method='ffill')
analysis_df['Tanker_Disruption_Pct'] = analysis_df['Tanker_Disruption_Pct'].fillna(method='ffill')

# Fill sentiment with neutral (0) when no crisis
analysis_df['sentiment_mean'] = analysis_df['sentiment_mean'].fillna(0)
analysis_df['article_count'] = analysis_df['article_count'].fillna(0)

# Calculate returns
for col in commodities_df.columns:
    analysis_df[f'{col}_Return'] = analysis_df[col].pct_change()

# Create crisis indicators
analysis_df['Shipping_Crisis'] = (analysis_df['Disruption_Pct'] < -20).astype(int)
analysis_df['Sentiment_Crisis'] = (analysis_df['sentiment_mean'] < -0.5).astype(int)
analysis_df['Combined_Crisis'] = ((analysis_df['Shipping_Crisis'] == 1) | 
                                   (analysis_df['Sentiment_Crisis'] == 1)).astype(int)

print(f"\nAnalysis DataFrame Shape: {analysis_df.shape}")
print(f"Date range: {analysis_df.index.min()} to {analysis_df.index.max()}")
print(f"\nCrisis Days:")
print(f"  Shipping Crisis: {analysis_df['Shipping_Crisis'].sum()} days")
print(f"  Sentiment Crisis: {analysis_df['Sentiment_Crisis'].sum()} days")
print(f"  Combined Crisis: {analysis_df['Combined_Crisis'].sum()} days")

analysis_df.head()

## 4. Commodity Response Analysis

In [ ]:
# Calculate average returns during different scenarios
scenarios = {
    'Normal': (analysis_df['Combined_Crisis'] == 0),
    'Shipping Crisis Only': ((analysis_df['Shipping_Crisis'] == 1) & (analysis_df['Sentiment_Crisis'] == 0)),
    'Sentiment Crisis Only': ((analysis_df['Shipping_Crisis'] == 0) & (analysis_df['Sentiment_Crisis'] == 1)),
    'Combined Crisis': ((analysis_df['Shipping_Crisis'] == 1) & (analysis_df['Sentiment_Crisis'] == 1))
}

results = {}
for scenario_name, mask in scenarios.items():
    scenario_data = analysis_df[mask]
    results[scenario_name] = {
        'Days': mask.sum(),
        'USO': scenario_data['USO_Return'].mean() * 252 * 100,
        'BNO': scenario_data['BNO_Return'].mean() * 252 * 100,
        'UNG': scenario_data['UNG_Return'].mean() * 252 * 100,
        'BDRY': scenario_data['BDRY_Return'].mean() * 252 * 100,
        'CL_Fut': scenario_data['CL_Fut_Return'].mean() * 252 * 100,
        'BZ_Fut': scenario_data['BZ_Fut_Return'].mean() * 252 * 100,
        'NG_Fut': scenario_data['NG_Fut_Return'].mean() * 252 * 100
    }

results_df = pd.DataFrame(results).T
print("\nAnnualized Returns by Scenario (%)")
print(results_df.round(2))

In [ ]:
# Visualize scenario comparison
fig, axes = plt.subplots(2, 2, figsize=(20, 14))

# Plot 1: Returns by scenario
results_df.drop('Days', axis=1).T.plot(kind='bar', ax=axes[0, 0], width=0.8)
axes[0, 0].set_title('Annualized Returns by Scenario', fontsize=14, fontweight='bold')
axes[0, 0].set_ylabel('Return (%)')
axes[0, 0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[0, 0].legend(title='Scenario', bbox_to_anchor=(1.05, 1), loc='upper left')
axes[0, 0].grid(True, alpha=0.3)
plt.setp(axes[0, 0].xaxis.get_majorticklabels(), rotation=45, ha='right')

# Plot 2: Crisis impact (difference from normal)
impact_df = results_df.drop('Days', axis=1).subtract(results_df.loc['Normal'].drop('Days'), axis=1)
impact_df.drop('Normal').T.plot(kind='bar', ax=axes[0, 1], width=0.8)
axes[0, 1].set_title('Crisis Impact (Difference from Normal)', fontsize=14, fontweight='bold')
axes[0, 1].set_ylabel('Return Difference (%)')
axes[0, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[0, 1].legend(title='Crisis Type', bbox_to_anchor=(1.05, 1), loc='upper left')
axes[0, 1].grid(True, alpha=0.3)
plt.setp(axes[0, 1].xaxis.get_majorticklabels(), rotation=45, ha='right')

# Plot 3: Oil products comparison
oil_cols = ['USO', 'BNO', 'CL_Fut', 'BZ_Fut']
results_df[oil_cols].T.plot(kind='bar', ax=axes[1, 0], width=0.8)
axes[1, 0].set_title('Oil Products Response by Scenario', fontsize=14, fontweight='bold')
axes[1, 0].set_ylabel('Annualized Return (%)')
axes[1, 0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1, 0].legend(title='Scenario', bbox_to_anchor=(1.05, 1), loc='upper left')
axes[1, 0].grid(True, alpha=0.3)
plt.setp(axes[1, 0].xaxis.get_majorticklabels(), rotation=45, ha='right')

# Plot 4: Gas and Shipping
other_cols = ['UNG', 'NG_Fut', 'BDRY']
results_df[other_cols].T.plot(kind='bar', ax=axes[1, 1], width=0.8)
axes[1, 1].set_title('Gas & Shipping Response by Scenario', fontsize=14, fontweight='bold')
axes[1, 1].set_ylabel('Annualized Return (%)')
axes[1, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1, 1].legend(title='Scenario', bbox_to_anchor=(1.05, 1), loc='upper left')
axes[1, 1].grid(True, alpha=0.3)
plt.setp(axes[1, 1].xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

## 5. Correlation Analysis

In [ ]:
# Calculate correlations
corr_cols = ['Disruption_Pct', 'Tanker_Disruption_Pct', 'sentiment_mean'] + \
            [f'{col}_Return' for col in commodities_df.columns]

corr_df = analysis_df[corr_cols].corr()

# Focus on correlations with disruption and sentiment
key_corr = corr_df.loc[['Disruption_Pct', 'Tanker_Disruption_Pct', 'sentiment_mean'], 
                        [f'{col}_Return' for col in commodities_df.columns]]

print("\nCorrelations with Returns:")
print(key_corr.round(3))

# Visualize correlation heatmap
fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(key_corr, annot=True, fmt='.3f', cmap='RdYlGn', center=0, 
            cbar_kws={'label': 'Correlation'}, ax=ax)
ax.set_title('Correlation: Disruption/Sentiment vs Commodity Returns', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Time Series Analysis During Crisis Events

In [ ]:
# Identify major crisis periods
crisis_events = sentiment_df.groupby('crisis_event').agg({
    'crisis_start': 'first',
    'crisis_end': 'first',
    'sentiment': 'mean',
    'date': 'count'
}).rename(columns={'date': 'article_count'})

print("\nCrisis Events:")
print(crisis_events)

# Analyze each crisis event
for event_name, event_data in crisis_events.iterrows():
    start = pd.to_datetime(event_data['crisis_start'])
    end = pd.to_datetime(event_data['crisis_end'])
    
    # Get data for event period
    event_mask = (analysis_df.index >= start) & (analysis_df.index <= end)
    event_df = analysis_df[event_mask]
    
    if len(event_df) == 0:
        continue
    
    print(f"\n{'='*80}")
    print(f"EVENT: {event_name}")
    print(f"Period: {start.date()} to {end.date()} ({len(event_df)} trading days)")
    print(f"{'='*80}")
    
    # Calculate cumulative returns
    cum_returns = {}
    for col in commodities_df.columns:
        if f'{col}_Return' in event_df.columns:
            cum_ret = (1 + event_df[f'{col}_Return']).prod() - 1
            cum_returns[col] = cum_ret * 100
    
    cum_returns_series = pd.Series(cum_returns).sort_values(ascending=False)
    print("\nCumulative Returns (%):")
    print(cum_returns_series.round(2))
    
    # Average shipping disruption
    avg_disruption = event_df['Disruption_Pct'].mean()
    print(f"\nAverage Shipping Disruption: {avg_disruption:.2f}%")
    print(f"Average Sentiment: {event_df['sentiment_mean'].mean():.3f}")

## 7. Key Insights

In [ ]:
print("\n" + "="*80)
print("KEY INSIGHTS - COMMODITIES RESPONSE TO SHIPPING + SENTIMENT")
print("="*80)

print("\n1. SCENARIO COMPARISON:")
print(f"   Total days analyzed: {len(analysis_df)}")
for scenario, data in results.items():
    print(f"   {scenario}: {data['Days']} days")

print("\n2. BEST PERFORMERS DURING COMBINED CRISIS:")
combined_crisis_returns = results_df.loc['Combined Crisis'].drop('Days').sort_values(ascending=False)
for ticker, ret in combined_crisis_returns.head(3).items():
    print(f"   {ticker}: {ret:+.2f}% annualized")

print("\n3. WORST PERFORMERS DURING COMBINED CRISIS:")
for ticker, ret in combined_crisis_returns.tail(3).items():
    print(f"   {ticker}: {ret:+.2f}% annualized")

print("\n4. CORRELATION INSIGHTS:")
print("   Strongest correlations with shipping disruption:")
ship_corr = key_corr.loc['Disruption_Pct'].abs().sort_values(ascending=False)
for ticker, corr in ship_corr.head(3).items():
    actual_corr = key_corr.loc['Disruption_Pct', ticker]
    print(f"   {ticker}: {actual_corr:+.3f}")

print("\n5. TRADING IMPLICATIONS:")
print("   During shipping disruptions:")
ship_only_impact = impact_df.loc['Shipping Crisis Only'].sort_values(ascending=False)
print(f"   - LONG: {ship_only_impact.head(2).index.tolist()}")
print(f"   - SHORT: {ship_only_impact.tail(2).index.tolist()}")

print("\n   During sentiment crises:")
sent_only_impact = impact_df.loc['Sentiment Crisis Only'].sort_values(ascending=False)
print(f"   - LONG: {sent_only_impact.head(2).index.tolist()}")
print(f"   - SHORT: {sent_only_impact.tail(2).index.tolist()}")

print("\n" + "="*80)